# 단어 BiLSTM NER — Word Segmentation 모범답안과 같은 골격 (실제 캐글 제출)

**연습 기법**: IOAI 2025 GAITE **Word Segmentation** 모범답안과 **같은 코드 골격**의 시퀀스 라벨링을 **실제 캐글 NER
대회**에 적용한다. 20번(글자 분할)과 코드 구조가 1:1 로 같고, 두 군데만 다르다:

| | 20번 (Word Seg 모범답안) | 21번 (이 노트북, NER) |
|---|---|---|
| 단위 | 글자 | 단어 |
| 입력 | one-hot (글자 어휘 작음) | **Embedding** (단어 어휘 큼) |
| 출력/손실 | sigmoid + **BCELoss** (0/1) | 로짓 + **CrossEntropyLoss** (7태그) |
| 나머지 | `CompoundDataset`·`collate_fn`·`MyModel`·`train()`·예측 | **동일 골격** |

**대회**: [MIPT NER](https://www.kaggle.com/c/mipt-ner) — 러시아어 개체명 인식(BIO 7태그). 20번과 달리 **실제 캐글
대회**라 `id,tag` 로 **진짜 CSV 제출**한다. 경량(dev 0.43MB).

> ⚙️ GPU 권장(작은 BiLSTM, T4 ~1분).  ⚠️ API 토큰 평문 — 외부 공유 금지.
> 🔑 **첫 실행 시 Rules 수락**: 403 이면 [대회 페이지](https://www.kaggle.com/c/mipt-ner/rules)에서 'I Understand and Accept' 1회.

## 0. 설치 · 자격증명 · 시드

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

import random, numpy as np, torch
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("준비 완료")

## 1. Kaggle 에서 데이터 다운로드 (dev / test / sample_submission)
403 이면 대회 Rules 를 한 번 수락(위 안내).

In [ ]:
import glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
try:
    api.competition_download_files("mipt-ner", path=WORK_DIR, quiet=False)
    for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
        with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
        os.remove(zp)
    print("다운로드 완료")
except Exception as e:
    print("⚠️ 다운로드 실패:", repr(e)[:160])
    print("→ 403 이면 https://www.kaggle.com/c/mipt-ner/rules 에서 'I Understand and Accept' 1회 후 재실행")

## 2. data · word2idx · tag2idx (20번의 char2idx 자리에 단어/태그 사전)
`dev.csv` 의 (태그열, 문장) → `(words, tags)` 리스트. 20번의 `(글자열, 라벨)` 과 같은 모양(단위만 단어). 학습/검증 분할.

In [ ]:
import pandas as pd
TAGS = ["O", "B_PER", "I_PER", "B_LOC", "I_LOC", "B_ORG", "I_ORG"]
tag2idx = {t: i for i, t in enumerate(TAGS)}

dev = pd.read_csv(os.path.join(WORK_DIR, "dev.csv")); dev.columns = ["tags", "text", "flag"]
alld = [(str(r.text).split(), str(r.tags).split()) for r in dev.itertuples()
        if len(str(r.text).split()) == len(str(r.tags).split())]
random.shuffle(alld)
cut = int(len(alld) * 0.85)
data, val_data = alld[:cut], alld[cut:]

# 단어 사전 (0=padding, 1=unknown)
word2idx = {"<pad>": 0, "<unk>": 1}
for words, _ in data:
    for w in words: word2idx.setdefault(w.lower(), len(word2idx))
vocab_size = len(word2idx)
print("data:", len(data), "| val:", len(val_data), "| vocab_size:", vocab_size, "| tags:", len(TAGS))
print("예시:", data[0][0][:6], "|", data[0][1][:6])

## 3. NERDataset & collate_fn (20번 CompoundDataset 과 같은 구조)
라벨이 0/1 실수 대신 **태그 정수**라 `dtype=long`, 패딩 라벨은 손실에서 마스크로 제외.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class NERDataset(Dataset):
    def __init__(self, data, word2idx):
        self.data = data
        self.word2idx = word2idx
    def __len__(self):
        return len(self.data)
    def encode(self, words, tags):
        return (
            torch.tensor([self.word2idx.get(w.lower(), 1) for w in words], dtype=torch.long),
            torch.tensor([tag2idx[t] for t in tags], dtype=torch.long),
        )
    def __getitem__(self, idx):
        words, tags = self.data[idx]
        return self.encode(words, tags)

def collate_fn(batch):
    inputs, targets = zip(*batch)
    lengths = [len(seq) for seq in inputs]
    max_len = max(lengths)
    padded_inputs = torch.zeros(len(inputs), max_len, dtype=torch.long)
    padded_targets = torch.zeros(len(targets), max_len, dtype=torch.long)
    for i, (seq, tgt) in enumerate(zip(inputs, targets)):
        padded_inputs[i, : len(seq)] = seq
        padded_targets[i, : len(tgt)] = tgt
    return padded_inputs, padded_targets, lengths

## 4. MyModel — Embedding → BiLSTM → fc (20번과 같은 골격, 출력만 7태그)
단어 어휘가 커서 one-hot 대신 **Embedding** 을 쓰고, fc 출력이 7태그라 sigmoid 없이 로짓을 낸다.

In [ ]:
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, vocab_size, n_tags, emb_dim=64, hidden_dim=128, num_layers=2):
        super(MyModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2, n_tags)
    def forward(self, x):                       # x: (batch, seq)
        e = self.embedding(x)
        lstm_out, _ = self.lstm(e)
        return self.fc(lstm_out)                # (batch, seq, n_tags)  로짓

## 5. train() — CrossEntropyLoss + 길이 마스크 (20번과 같은 흐름)

In [ ]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train():
    dataset = NERDataset(data, word2idx)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
    model = MyModel(vocab_size, len(TAGS)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    num_epochs = 25
    for epoch in range(num_epochs):
        model.train(); epoch_loss = 0
        for inputs, targets, lengths in dataloader:
            targets = targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs.to(device))          # (batch, seq, n_tags)
            mask = torch.arange(inputs.shape[1])[None, :] < torch.tensor(lengths)[:, None]
            mask = mask.to(device)
            loss = criterion(outputs[mask], targets[mask])   # (N, n_tags) vs (N,)
            loss.backward(); optimizer.step(); epoch_loss += loss.item()
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(dataloader):.4f}")
    return model

model = train()

## 6. predict (20번 predict_and_save 와 같은 흐름) — 검증 F1
한 문장씩 태그를 예측한다(argmax). 라벨이 있는 검증셋으로 개체 F1 을 본다.

In [ ]:
from sklearn.metrics import f1_score

def predict(model, words):
    indices = [word2idx.get(w.lower(), 1) for w in words]
    x = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(x)[0].argmax(-1).cpu().numpy()
    return [TAGS[p] for p in pred]

model.eval(); P, G = [], []
for words, tags in val_data:
    P += [tag2idx[t] for t in predict(model, words)]; G += [tag2idx[t] for t in tags]
ENT = [tag2idx[t] for t in TAGS if t != "O"]
print("val 개체 micro-F1 =", round(f1_score(G, P, labels=ENT, average="micro", zero_division=0), 4))

## 7. 캐글 제출 — test 문장 태깅 → `id,tag` (공백 결합)
`test_res.csv` 는 헤더 없어 `header=None`. 각 문장을 태깅해 공백으로 이어 제출(샘플과 동일 형식).

In [ ]:
test = pd.read_csv(os.path.join(WORK_DIR, "test_res.csv"), header=None)
rows = [" ".join(predict(model, str(test.iloc[i, 0]).split())) for i in range(len(test))]

submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": np.arange(len(rows)), "tag": rows}).to_csv(submission_path, index=False)
sample = pd.read_csv(os.path.join(WORK_DIR, "crf_sample_submission.csv"))
ok = (len(rows) == len(sample)) and all(len(str(test.iloc[i,0]).split()) == len(rows[i].split()) for i in range(len(rows)))
print("Saved:", submission_path, "| rows:", len(rows), "| 형식 OK:", ok)
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files; files.download(submission_path)
except Exception as e:
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [MIPT NER](https://www.kaggle.com/c/mipt-ner/submit) 에 제출. **20번(Word Segmentation 모범답안
골격)과 같은 구조**(`Dataset`→`collate_fn`→`MyModel`→`train()`→예측)를 NER 에 적용한 *실제 캐글 제출* 버전.
데이터가 적어(1.3천 문장) 개체 F1 은 낮은 편 — 더 올리려면(같은 골격에 추가): char-CNN/대문자 특징(OOV 처리), CRF 디코딩, dropout, 사전학습/다국어 임베딩.